# Demo cấp 1 · CLIP + GAT trên scene graph Visual Genome

Nhập câu (hoặc chọn một trong 10 ca của báo cáo) → top-k ảnh sau khi xếp hạng lại → đồ thị giải thích S của từng ảnh (các quan hệ của scene graph làm tăng điểm đồ thị nhiều nhất, K* đọc từ kết quả notebook 11) → kiểm chứng: bỏ S hoặc bỏ cùng số quan hệ ngẫu nhiên rồi xếp hạng lại. Mô hình là checkpoint chính đã khóa; không huấn luyện, không chọn gì trên test.

**Cách dùng khi quay video:** Runtime → Run all. Ô cuối in ra một link `https://….gradio.live`: mở link ở tab mới (toàn màn hình).
Chọn ca trong danh sách là trang tự chạy; bấm **Bỏ giải thích S** trên đồ thị để xem ảnh rơi hạng. Link công khai và tự hết hạn,
tắt notebook (Runtime → Disconnect) sau khi quay. Đặt biến môi trường `XAI_DEMO_SHARE=0` nếu chỉ muốn xem trong notebook.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'torch-geometric', 'pyarrow', 'pandas', 'scipy', 'networkx', 'open_clip_torch', 'requests', 'gradio'])
print('Đã cài thư viện')

## Cấu hình, dữ liệu, mô hình (giống notebook 11)

In [ ]:
import os, json, glob, time, random, math, hashlib
from pathlib import Path
import numpy as np, pandas as pd, pyarrow.parquet as pq
import torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import GATConv

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Không chạy trên Colab:', exc)

def find_final_root():
    env = os.environ.get('FINAL_ROOT')
    if env and Path(env).is_dir():
        return Path(env)
    pats = ['/content/drive/MyDrive/MLGs - Project/0. final',
            '/content/drive/MyDrive/*/MLGs - Project/0. final',
            '/content/drive/MyDrive/*/*/MLGs - Project/0. final',
            '/content/drive/Shareddrives/*/MLGs - Project/0. final']
    for p in pats:
        hits = sorted(glob.glob(p))
        if hits:
            return Path(hits[0])
    raise FileNotFoundError('Không tìm thấy "MLGs - Project/0. final". Thêm lối tắt vào My Drive hoặc đặt FINAL_ROOT.')

FINAL = find_final_root()
DATA_ROOT = FINAL / 'data/VisualGenome/processed'
RESULTS_ROOT = FINAL / 'results/VisualGenome'
GRAPH_ROOT, CLIP_ROOT = DATA_ROOT / 'graph', DATA_ROOT / 'clip'
FEATURE_ROOT = CLIP_ROOT / 'features'
OUT = RESULTS_ROOT / 'XAI'
(OUT / 'cases').mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MAIN_SEED = 2026
SEEDS = [2026, 2027, 2028]
SELECTION_SEED = 2026          # bốc truy vấn cố định, không nhìn kết quả
N_VAL_K = 300                  # số truy vấn val để chọn K*
N_TEST = 500                   # số truy vấn test đo fidelity mỗi seed
N_RANDOM = 10                  # số lần bỏ K đơn vị ngẫu nhiên
K_GRID = [1, 2, 3, 5, 8, 12, 16]
N_CASE_POOL = 20000            # số truy vấn test dùng để chọn ca 5 + 5
RUN_FULL_REPLAY = True
print('FINAL =', FINAL, '| DEVICE =', DEVICE)

In [ ]:
compact = torch.load(DATA_ROOT / 'graph_cache/compact_graph_cache_v2.pt', map_location='cpu', weights_only=False)
assert compact['node_ptr'][-1].item() == 2_609_364 and compact['edge_ptr'][-1].item() == 8_005_044
image_features = torch.load(FEATURE_ROOT / 'image_features.pt', map_location='cpu', weights_only=True).float()
entity_features = torch.load(FEATURE_ROOT / 'entity_label_features.pt', map_location='cpu', weights_only=True).float()
relation_features = torch.load(FEATURE_ROOT / 'relation_text_features.pt', map_location='cpu', weights_only=True).float()
relation_mapping = pd.read_parquet(GRAPH_ROOT / 'relation_mapping.parquet')
relation_feature_index = pd.read_parquet(FEATURE_ROOT / 'relation_feature_index.parquet')
relation_join = relation_mapping.merge(relation_feature_index[['edge_family', 'relation', 'feature_row']],
                                       on=['edge_family', 'relation'], how='left', validate='one_to_one')
relation_type_features = torch.empty((int(relation_mapping.relation_index.max()) + 1, 512), dtype=torch.float32)
known = relation_join.feature_row.notna().to_numpy()
relation_type_features[torch.from_numpy(relation_join.loc[known, 'relation_index'].to_numpy(np.int64))] = \
    relation_features[torch.from_numpy(relation_join.loc[known, 'feature_row'].to_numpy(np.int64))]
unk_rows = relation_feature_index.loc[relation_feature_index.learnable_relation_index == 1, 'feature_row'].to_numpy(np.int64)
unk_feature = F.normalize(relation_features[torch.from_numpy(unk_rows)].mean(0), dim=0)
missing = relation_join.loc[~known, 'relation_index'].to_numpy(np.int64)
assert set(missing).issubset({1})
if len(missing):
    relation_type_features[torch.from_numpy(missing)] = unk_feature
REL_NAME = dict(zip(relation_mapping.relation_index, relation_mapping.relation))

image_index = pd.read_parquet(FEATURE_ROOT / 'image_feature_index.parquet', columns=['image_id', 'feature_row'])
image_to_component = dict(zip(image_index.image_id.astype(str), image_index.feature_row))

def pick_text_col(df, prefer):
    cols = [c for c in df.columns
            if pd.api.types.is_string_dtype(df[c]) or pd.api.types.is_object_dtype(df[c])]
    assert cols, f'Không có cột chữ trong {list(df.columns)}'
    pref = [c for c in cols if any(k in c.lower() for k in prefer)]
    return (pref or cols)[0]

efi = pd.read_parquet(FEATURE_ROOT / 'entity_feature_index.parquet')
print('entity_feature_index:', list(efi.columns))
ENT_LABEL_COL = pick_text_col(efi, ('label', 'name', 'canonical', 'text'))
ENT_LABEL = (dict(zip(efi.feature_row, efi[ENT_LABEL_COL])) if 'feature_row' in efi.columns
             else dict(enumerate(efi[ENT_LABEL_COL])))
print('Dùng cột tên thực thể:', ENT_LABEL_COL, '| ví dụ:', list(ENT_LABEL.values())[:8])

def node_label(g, v):
    return 'ẢNH' if v == g.root else str(ENT_LABEL.get(int(g.feature_rows[v]), f'thực thể {v}'))

def load_queries(split):
    q = pd.read_parquet(CLIP_ROOT / split / 'queries.parquet')
    col = pick_text_col(q.drop(columns=[c for c in q.columns if c.endswith('_id')], errors='ignore'),
                        ('text', 'phrase', 'caption', 'query', 'description'))
    return dict(zip(q.text_id.astype(str), q[col].astype(str)))

IMG_URL = {}
try:
    imgs = pd.read_parquet(DATA_ROOT / 'splits/images.parquet')
    ucol = [c for c in imgs.columns if 'url' in c.lower()]
    if ucol:
        IMG_URL = dict(zip(imgs.image_id.astype(str), imgs[ucol[0]].astype(str)))
    print('images.parquet:', list(imgs.columns), '| có URL ảnh:', bool(ucol))
except Exception as exc:
    print('Không đọc được URL ảnh (chỉ vẽ đồ thị):', exc)
print('Đã nạp đặc trưng')

In [ ]:
class EdgeAwareGAT(nn.Module):
    # Giữ nguyên kiến trúc của run_gat.py / evaluate_test.ipynb
    def __init__(self, input_dim=512, hidden_dim=128, heads=4, dropout=0.2):
        super().__init__()
        self.dropout = dropout
        self.node_projection = nn.Linear(input_dim, hidden_dim)
        self.query_projection = nn.Linear(input_dim, hidden_dim)
        self.edge_projection = nn.Linear(input_dim + 1, hidden_dim)
        self.type_embedding = nn.Embedding(2, hidden_dim)
        self.gat1 = GATConv(hidden_dim, hidden_dim // heads, heads=heads, edge_dim=hidden_dim, dropout=dropout, add_self_loops=False)
        self.gat2 = GATConv(hidden_dim, hidden_dim // heads, heads=heads, edge_dim=hidden_dim, dropout=dropout, add_self_loops=False)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.logit_scale = nn.Parameter(torch.tensor(2.6592))

    def encode_graph(self, batch):
        h = self.node_projection(batch['x']) + self.type_embedding(batch['node_type'])
        edge_attr = self.edge_projection(torch.cat([batch['edge_attr'], batch['edge_reverse']], dim=1))
        update = self.gat1(h, batch['edge_index'], edge_attr=edge_attr)
        h = self.norm1(h + F.dropout(F.elu(update), self.dropout, self.training))
        update = self.gat2(h, batch['edge_index'], edge_attr=edge_attr)
        h = self.norm2(h + F.dropout(F.elu(update), self.dropout, self.training))
        return F.normalize(h[batch['root_local']], dim=1)

    def encode_query(self, q):
        return F.normalize(self.query_projection(q), dim=1)

RUNS = {
    'seed_2026': ('3SEED/seed_2026', 'real_edges'),
    'seed_2027': ('3SEED/seed_2027', 'real_edges'),
    'seed_2028': ('3SEED/seed_2028', 'real_edges'),
    'no_edge': ('ABLATION/no_edge_seed_2026', 'no_edge'),
    'random_edges': ('ABLATION/random_edges_seed_2026', 'random_edges'),
}
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(8 << 20), b''):
            h.update(b)
    return h.hexdigest()

MODELS = {}
for name, (rel, mode) in RUNS.items():
    path = RESULTS_ROOT / rel / 'best_gat.pt'
    ck = torch.load(path, map_location='cpu', weights_only=False)
    assert ck.get('edge_mode', 'real_edges') == mode, (name, ck.get('edge_mode'))
    m = EdgeAwareGAT().to(DEVICE)
    m.load_state_dict(ck['model_state_dict'])
    m.eval()
    MODELS[name] = {'model': m, 'alpha': float(ck['alpha']), 'epoch': int(ck['epoch']), 'edge_mode': mode,
                    'sha256': sha256(path)}
    print(f"{name}: epoch {ck['epoch']}, α_CLIP = {ck['alpha']}, edge_mode = {mode}")

In [ ]:
# Lõi bộ giải thích cấp 1 cho CLIP + EdgeAwareGAT trên Visual Genome.
# Được nhúng nguyên văn vào notebook 11_visualgenome_gat_xai_level1.ipynb (ô "Lõi bộ giải thích").
#
# Đơn vị giải thích = một quan hệ của scene graph trong ảnh: cạnh gốc và cạnh ngược của nó
# (cùng hai đầu, cùng loại quan hệ) gộp thành một đơn vị. Cạnh ảnh -> thực thể ("contains") cũng là đơn vị.
# Đóng góp của đơn vị u: delta_u = S_full - S_(full \ u), trong đó S là điểm đồ thị q'.z của ảnh và
# z được tính lại bằng chính 2 tầng GAT của checkpoint sau khi bỏ u (giống bộ giải thích của notebook 06).
import math
import random

import numpy as np
import torch
import torch.nn.functional as F


class ComponentGraph:
    """Đồ thị của một ảnh (một thành phần liên thông trong compact cache), đặt sẵn trên thiết bị."""

    def __init__(self, compact, comp, image_features, entity_features, relation_type_features,
                 device, edge_mode="real_edges", random_edge_seed=2026):
        ns, ne = compact["node_ptr"][comp:comp + 2].tolist()
        es, ee = compact["edge_ptr"][comp:comp + 2].tolist()
        nodes = compact["node_perm"][ns:ne]
        edges = compact["edge_index"][:, es:ee].clone()
        if edge_mode == "random_edges":
            gen = torch.Generator().manual_seed(int(random_edge_seed) + int(comp))
            perm = torch.randperm(edges.shape[1], generator=gen)
            edges = torch.stack((edges[0], edges[1, perm]))
        g2l = {int(n): i for i, n in enumerate(nodes.tolist())}
        src = torch.tensor([g2l[int(v)] for v in edges[0].tolist()], dtype=torch.long)
        dst = torch.tensor([g2l[int(v)] for v in edges[1].tolist()], dtype=torch.long)
        node_type = compact["node_type"][nodes].long()
        roots = (node_type == 1).nonzero().flatten()
        assert roots.numel() == 1, "mỗi thành phần phải có đúng một nút ảnh"
        feat_rows = compact["node_feature_row"][nodes]
        x = torch.empty((nodes.numel(), image_features.shape[1]), dtype=torch.float32)
        img = node_type == 1
        x[img] = image_features[feat_rows[img]]
        x[~img] = entity_features[feat_rows[~img]]
        rel = compact["edge_relation_type"][es:ee].long()
        rev = compact["edge_is_reverse"][es:ee].float()

        self.comp = int(comp)
        self.device = device
        self.edge_mode = edge_mode
        self.nodes_global = nodes
        self.feature_rows = feat_rows
        self.node_type = node_type.to(device)
        self.x = x.to(device)
        self.root = int(roots.item())
        self.src, self.dst = src, dst
        self.rel = rel
        self.rev = rev
        self.edge_attr = relation_type_features[rel].to(device)
        self.edge_reverse = rev.unsqueeze(1).to(device)
        self.n_nodes = int(nodes.numel())
        self.n_edges = int(src.numel())
        # Gộp cạnh gốc và cạnh ngược thành đơn vị quan hệ.
        groups = {}
        for e in range(self.n_edges):
            a, b = int(src[e]), int(dst[e])
            key = (min(a, b), max(a, b), int(rel[e]))
            groups.setdefault(key, []).append(e)
        self.unit_keys = list(groups.keys())
        self.unit_edges = [groups[k] for k in self.unit_keys]
        self.n_units = len(self.unit_keys)
        um = torch.zeros((self.n_units, self.n_edges), dtype=torch.bool)
        for u, es_ in enumerate(self.unit_edges):
            um[u, es_] = True
        self.unit_edge_mask = um
        # Khoảng cách tới nút ảnh (vô hướng) để kiểm tra vùng tiếp nhận 2 tầng.
        adj = [[] for _ in range(self.n_nodes)]
        for a, b in zip(src.tolist(), dst.tolist()):
            adj[a].append(b)
            adj[b].append(a)
        dist = [math.inf] * self.n_nodes
        dist[self.root] = 0
        frontier = [self.root]
        while frontier:
            nxt = []
            for v in frontier:
                for w in adj[v]:
                    if dist[w] == math.inf:
                        dist[w] = dist[v] + 1
                        nxt.append(w)
            frontier = nxt
        self.dist = dist
        self.unit_is_contains = [k[2] == 0 for k in self.unit_keys]

    def unit_in_receptive_field(self, u):
        a, b, _ = self.unit_keys[u]
        return min(self.dist[a], self.dist[b]) <= 1

    def keep_mask_without(self, units):
        keep = torch.ones(self.n_edges, dtype=torch.bool)
        if len(units):
            keep &= ~self.unit_edge_mask[list(units)].any(0)
        return keep

    def keep_mask_only(self, units):
        if len(units) == 0:
            return torch.zeros(self.n_edges, dtype=torch.bool)
        return self.unit_edge_mask[list(units)].any(0)


def _encode_batched(model, g, masks, edge_index_override=None):
    """Mã hóa nút ảnh cho nhiều biến thể của cùng một đồ thị trong một lần chạy.

    masks: bool [M, E] (cạnh được giữ). edge_index_override: list độ dài M gồm (src, dst) thay thế hoặc None.
    """
    M = masks.shape[0]
    n = g.n_nodes
    xs, types, eis, eas, ers, roots = [], [], [], [], [], []
    for m in range(M):
        keep = masks[m]
        if edge_index_override is not None and edge_index_override[m] is not None:
            s, d = edge_index_override[m]
        else:
            s, d = g.src, g.dst
        ei = torch.stack((s[keep], d[keep])) + m * n
        eis.append(ei)
        eas.append(g.edge_attr[keep.to(g.device)])
        ers.append(g.edge_reverse[keep.to(g.device)])
        roots.append(g.root + m * n)
    x = g.x.repeat(M, 1)
    nt = g.node_type.repeat(M)
    batch = {
        "x": x,
        "node_type": nt,
        "edge_index": torch.cat(eis, 1).to(g.device),
        "edge_attr": torch.cat(eas, 0),
        "edge_reverse": torch.cat(ers, 0),
        "root_local": torch.tensor(roots, device=g.device),
    }
    if g.edge_mode == "no_edge":
        batch["edge_index"] = batch["edge_index"][:, :0]
        batch["edge_attr"] = batch["edge_attr"][:0]
        batch["edge_reverse"] = batch["edge_reverse"][:0]
    return model.encode_graph(batch)


@torch.no_grad()
def graph_scores(model, g, qz, masks, edge_index_override=None, chunk=64):
    """Điểm đồ thị q'.z của ảnh cho từng biến thể (mask)."""
    out = []
    for i in range(0, masks.shape[0], chunk):
        ov = None if edge_index_override is None else edge_index_override[i:i + chunk]
        z = _encode_batched(model, g, masks[i:i + chunk], ov)
        out.append(z @ qz)
    return torch.cat(out).float().cpu()


@torch.no_grad()
def unit_deltas(model, g, qz):
    """delta_u = S_full - S_(full \\ u) cho mọi đơn vị; trả về (S_full, S_none, delta[U])."""
    full = torch.ones((1, g.n_edges), dtype=torch.bool)
    none = torch.zeros((1, g.n_edges), dtype=torch.bool)
    leave = ~g.unit_edge_mask  # [U, E]: bỏ đúng một đơn vị
    s = graph_scores(model, g, qz, torch.cat([full, none, leave], 0))
    return float(s[0]), float(s[1]), (s[0] - s[2:])


def explanation(deltas, K):
    """S = tối đa K đơn vị có delta dương lớn nhất."""
    order = torch.argsort(deltas, descending=True).tolist()
    return [u for u in order[:K] if deltas[u] > 0]


@torch.no_grad()
def attention_unit_scores(model, g):
    """Điểm attention của từng đơn vị (tổng attention trung bình các head, cả hai tầng). Chỉ làm đường cơ sở."""
    ei = torch.stack((g.src, g.dst)).to(g.device)
    h = model.node_projection(g.x) + model.type_embedding(g.node_type)
    ea = model.edge_projection(torch.cat([g.edge_attr, g.edge_reverse], 1))
    upd, (ei1, a1) = model.gat1(h, ei, edge_attr=ea, return_attention_weights=True)
    h = model.norm1(h + F.elu(upd))
    _, (ei2, a2) = model.gat2(h, ei, edge_attr=ea, return_attention_weights=True)
    # GATConv không tự thêm self-loop (add_self_loops=False) nên thứ tự cạnh giữ nguyên.
    assert ei1.shape[1] == g.n_edges and ei2.shape[1] == g.n_edges
    per_edge = (a1.mean(1) + a2.mean(1)).cpu()
    return torch.stack([per_edge[e].sum() for e in g.unit_edges])


def rewired_edges(g, units, rng):
    """Làm nhiễu S: thay đầu không phải nút ảnh của mỗi cạnh trong S bằng một thực thể ngẫu nhiên khác của ảnh."""
    s, d = g.src.clone(), g.dst.clone()
    entities = [i for i in range(g.n_nodes) if i != g.root]
    for u in units:
        a, b, _ = g.unit_keys[u]
        for e in g.unit_edges[u]:
            # giữ đầu gần nút ảnh hơn, thay đầu còn lại
            if g.dist[int(s[e])] <= g.dist[int(d[e])]:
                cur = int(d[e])
                pool = [v for v in entities if v != cur and v != int(s[e])]
                if pool:
                    d[e] = rng.choice(pool)
            else:
                cur = int(s[e])
                pool = [v for v in entities if v != cur and v != int(d[e])]
                if pool:
                    s[e] = rng.choice(pool)
    return s, d


def fuse(clip_scores, graph_scores_vec, alpha):
    cz = (clip_scores - clip_scores.mean()) / clip_scores.std().clamp_min(1e-6)
    gz = (graph_scores_vec - graph_scores_vec.mean()) / graph_scores_vec.std().clamp_min(1e-6)
    return alpha * cz + (1.0 - alpha) * gz


def prob_rank(clip_scores, graph_vec, alpha, i, new_graph_score=None):
    """Xác suất softmax và hạng của ứng viên i khi điểm đồ thị của i được thay bằng new_graph_score."""
    gv = graph_vec.clone()
    if new_graph_score is not None:
        gv[i] = new_graph_score
    fused = fuse(clip_scores, gv, alpha)
    p = torch.softmax(fused, 0)[i].item()
    rank = 1 + int((fused > fused[i]).sum())
    return p, rank


def characterization(fid_plus, fid_minus):
    a = min(max(fid_plus, 0.0), 1.0)
    b = 1.0 - min(max(fid_minus, 0.0), 1.0)
    return 0.0 if a + b == 0 else 2 * a * b / (a + b)


def tokens(text):
    import re
    return set(w for w in re.findall(r"[a-z]+", str(text).lower()) if len(w) > 2)


def unit_grounded(g, u, query_tokens, node_label):
    """Đơn vị chạm một thực thể có từ xuất hiện trong câu truy vấn (validity theo nội dung)."""
    a, b, _ = g.unit_keys[u]
    for v in (a, b):
        if v == g.root:
            continue
        if tokens(node_label(g, v)) & query_tokens:
            return True
    return False


@torch.no_grad()
def explain_and_measure(model, g, qz, clip_scores, graph_vec, alpha, i, K, n_random=10, seed=2026,
                        query_tokens=None, node_label=None):
    """Đo đủ bộ chỉ số cho giải thích của ứng viên i (ảnh hạng 1 trả về)."""
    rng = random.Random(seed)
    s_full, s_none, deltas = unit_deltas(model, g, qz)
    S = explanation(deltas, K)
    U = g.n_units
    k = len(S)
    p0, r0 = prob_rank(clip_scores, graph_vec, alpha, i)
    out = {"S_full": s_full, "S_none": s_none, "n_units": U, "size_S": k, "p0": p0, "rank0": r0,
           "S": S, "deltas_S": [float(deltas[u]) for u in S]}
    if k == 0:
        out["explained"] = False
        return out
    out["explained"] = True
    masks = [g.keep_mask_without(S), g.keep_mask_only(S)]
    rand_sets = [rng.sample(range(U), k) for _ in range(n_random)]
    masks += [g.keep_mask_without(R) for R in rand_sets]
    masks += [g.keep_mask_only(R) for R in rand_sets]
    att = attention_unit_scores(model, g)
    A = torch.argsort(att, descending=True).tolist()[:k]
    masks += [g.keep_mask_without(A)]
    sc = graph_scores(model, g, qz, torch.stack(masks))
    s_rm, s_only = float(sc[0]), float(sc[1])
    s_rand_rm = sc[2:2 + n_random]
    s_rand_only = sc[2 + n_random:2 + 2 * n_random]
    s_att_rm = float(sc[-1])
    p_rm, r_rm = prob_rank(clip_scores, graph_vec, alpha, i, s_rm)
    p_only, r_only = prob_rank(clip_scores, graph_vec, alpha, i, s_only)
    p_att, r_att = prob_rank(clip_scores, graph_vec, alpha, i, s_att_rm)
    pr = [prob_rank(clip_scores, graph_vec, alpha, i, float(v)) for v in s_rand_rm]
    po = [prob_rank(clip_scores, graph_vec, alpha, i, float(v)) for v in s_rand_only]
    # làm nhiễu: thay đầu kia
    s_pert, d_pert = rewired_edges(g, S, rng)
    s_perturb = float(graph_scores(model, g, qz, torch.ones((1, g.n_edges), dtype=torch.bool),
                                   edge_index_override=[(s_pert, d_pert)])[0])
    p_pert, r_pert = prob_rank(clip_scores, graph_vec, alpha, i, s_perturb)
    fp = (p0 - p_rm) / p0
    fm = (p0 - p_only) / p0
    out.update({
        "fid_plus": fp,
        "fid_plus_rand": float(np.mean([(p0 - p) / p0 for p, _ in pr])),
        "fid_plus_attn": (p0 - p_att) / p0,
        "fid_minus": fm,
        "fid_minus_rand": float(np.mean([(p0 - p) / p0 for p, _ in po])),
        "characterization": characterization(fp, fm),
        "fid_perturb": (p0 - p_pert) / p0,
        "dscore_rm": s_full - s_rm,
        "dscore_rand": float(np.mean([s_full - float(v) for v in s_rand_rm])),
        "drank_rm": r_rm - r0,
        "drank_rand": float(np.mean([r - r0 for _, r in pr])),
        "drank_attn": r_att - r0,
        "PN": float(r_rm > 1),
        "PN_rand": float(np.mean([r > 1 for _, r in pr])),
        "PN_attn": float(r_att > 1),
        "PN_perturb": float(r_pert > 1),
        "PS": float(r_only == 1),
        "PS_rand": float(np.mean([r == 1 for _, r in po])),
        "sparsity": 1.0 - k / U,
        "in_receptive_field": float(np.mean([g.unit_in_receptive_field(u) for u in S])),
        "share_contains": float(np.mean([g.unit_is_contains[u] for u in S])),
        "win_rate": float(np.mean([(p0 - p_rm) > (p0 - p) for p, _ in pr])),
    })
    if query_tokens is not None and node_label is not None:
        out["ground_S"] = float(np.mean([unit_grounded(g, u, query_tokens, node_label) for u in S]))
        out["ground_rand"] = float(np.mean([np.mean([unit_grounded(g, u, query_tokens, node_label) for u in R])
                                            for R in rand_sets]))
        out["ground_base"] = float(np.mean([unit_grounded(g, u, query_tokens, node_label) for u in range(U)]))
    return out


@torch.no_grad()
def completeness_curve(model, g, qz, Ks):
    """D(K)/D_all: mức giảm điểm đồ thị khi bỏ top-K so với bỏ mọi đơn vị có delta dương (để chọn K* trên val)."""
    s_full, _, deltas = unit_deltas(model, g, qz)
    pos = [u for u in torch.argsort(deltas, descending=True).tolist() if deltas[u] > 0]
    if not pos:
        return None
    masks = [g.keep_mask_without(pos)] + [g.keep_mask_without(pos[:K]) for K in Ks]
    sc = graph_scores(model, g, qz, torch.stack(masks))
    d_all = s_full - float(sc[0])
    if d_all <= 1e-9:
        return None
    return {K: (s_full - float(sc[1 + j])) / d_all for j, K in enumerate(Ks)}

In [ ]:
global_to_local = torch.full((2_609_364,), -1, dtype=torch.long)

def build_component_batch(component_rows, edge_mode):
    node_chunks, edge_chunks, rel_chunks, rev_chunks, roots = [], [], [], [], []
    for comp in component_rows:
        ns, ne = compact['node_ptr'][comp:comp + 2].tolist()
        es, ee = compact['edge_ptr'][comp:comp + 2].tolist()
        nodes = compact['node_perm'][ns:ne]
        edges = compact['edge_index'][:, es:ee]
        if edge_mode == 'random_edges':
            gen = torch.Generator().manual_seed(2026 + comp)
            perm = torch.randperm(edges.shape[1], generator=gen)
            edges = torch.stack((edges[0], edges[1, perm]))
        nt = compact['node_type'][nodes]
        roots.append(nodes[nt == 1].item())
        node_chunks.append(nodes); edge_chunks.append(edges)
        rel_chunks.append(compact['edge_relation_type'][es:ee]); rev_chunks.append(compact['edge_is_reverse'][es:ee])
    ng = torch.cat(node_chunks); eg = torch.cat(edge_chunks, 1)
    global_to_local[ng] = torch.arange(ng.numel())
    el = global_to_local[eg]; rl = global_to_local[torch.tensor(roots)]
    nt = compact['node_type'][ng].long(); fr = compact['node_feature_row'][ng]
    x = torch.empty((ng.numel(), 512)); im = nt == 1
    x[im] = image_features[fr[im]]; x[~im] = entity_features[fr[~im]]
    ea = relation_type_features[torch.cat(rel_chunks)]
    er = torch.cat(rev_chunks).float().unsqueeze(1)
    if edge_mode == 'no_edge':
        el, ea, er = el[:, :0], ea[:0], er[:0]
    global_to_local[ng] = -1
    return {'x': x.to(DEVICE), 'node_type': nt.to(DEVICE), 'edge_index': el.to(DEVICE),
            'edge_attr': ea.to(DEVICE), 'edge_reverse': er.to(DEVICE), 'root_local': rl.to(DEVICE)}

@torch.no_grad()
def encode_candidates(name, image_ids, bs=128):
    spec = MODELS[name]
    comps = [image_to_component[x] for x in image_ids]
    out = [spec['model'].encode_graph(build_component_batch(comps[i:i + bs], spec['edge_mode'])).cpu()
           for i in range(0, len(comps), bs)]
    return torch.cat(out)

SPLIT = {}
def load_split(split):
    root = CLIP_ROOT / split
    cand = pd.read_parquet(root / 'candidate_images.parquet', columns=['image_id']).image_id.astype(str).tolist()
    res = pq.read_table(root / 'retrieval_results.parquet',
                        columns=['text_id', 'best_positive_rank', 'topk_image_ids', 'topk_scores', 'positive_image_id']).to_pandas()
    ti = pd.read_parquet(root / 'text_embedding_index.parquet')
    trow = dict(zip(ti.text_id.astype(str), ti.embedding_row))
    tfeat = torch.load(root / 'text_embeddings.pt', map_location='cpu', weights_only=True)
    SPLIT[split] = {'cand': cand, 'row': {x: i for i, x in enumerate(cand)}, 'res': res, 'trow': trow,
                    'tfeat': tfeat, 'text': load_queries(split), 'emb': {}}
    print(split, ':', len(cand), 'ảnh,', len(res), 'truy vấn')

def split_emb(split, name):
    S = SPLIT[split]
    if name not in S['emb']:
        S['emb'][name] = encode_candidates(name, S['cand'])
    return S['emb'][name]

@torch.no_grad()
def score_row(split, name, row):
    # Trả về: ids (50), clip_scores, graph_vec, qz, fused, hạng ảnh đúng, chỉ số ảnh hạng 1
    S, spec = SPLIT[split], MODELS[name]
    q = S['tfeat'][S['trow'][str(row.text_id)]].float().unsqueeze(0).to(DEVICE)
    qz = spec['model'].encode_query(q)[0]
    ids = [str(x) for x in row.topk_image_ids]
    emb = split_emb(split, name)
    gv = (emb[[S['row'][x] for x in ids]] @ qz.cpu()).float()
    clip = torch.tensor(row.topk_scores, dtype=torch.float32)
    fused = fuse(clip, gv, spec['alpha'])
    pos = str(row.positive_image_id)
    rank = 1 + int((fused > fused[ids.index(pos)]).sum()) if pos in ids else int(row.best_positive_rank)
    clip_rank = (ids.index(pos) + 1) if pos in ids else int(row.best_positive_rank)
    return ids, clip, gv, qz, fused, rank, clip_rank, int(torch.argmax(fused))

load_split('val'); load_split('test')

In [ ]:
# Hình giải thích cấp 1, cùng kiểu với hình ca cấp 2 (07 / 18.py):
# ảnh | đồ thị giải thích (spring layout, nét xanh dương = quan hệ trong S, độ dày và số trong ngoặc = Δ điểm đồ thị).
# Chỉ vẽ đúng tập giải thích S. Nút trùng tên được gộp (×n); quan hệ trùng (đầu, quan hệ, đuôi) cộng Δ.
import math
import textwrap
from collections import OrderedDict

import matplotlib.pyplot as plt
import networkx as nx

# Cùng bảng màu với các biểu đồ của báo cáo (report/figures/scripts/make_figures.py):
# xanh dương = tập giải thích S, cam = ảnh/thực thể được trả về, xanh ngọc = thực thể neo, xám = nút khác.
C_EDGE, C_IMG, C_ANCHOR, C_OTHER = "#2a78d6", "#eb6834", "#1baf7a", "#d9d8d4"
C_LINK, C_QUERY = "#9cc0ef", "#52514e"
ROOT = "ẢNH"
# Cỡ chữ (pt) cho hình rộng FIG_W inch; khi chèn bằng \textwidth (7.16 in) chữ còn khoảng 6 đến 8 pt.
FS = dict(sup=11, title=10, node=9, edge=8.5, tick=9, label=9)
FIG_W, FIG_H = 8.0, 3.6


def _endpoints(g, u):
    """Chiều gốc của quan hệ (đầu, đuôi) theo chú thích Visual Genome."""
    e = g.unit_edges[u][0]
    s, d = int(g.src[e]), int(g.dst[e])
    if bool(g.rev[e]):
        s, d = d, s
    return s, d


def _rel(g, u, rel_name):
    r = rel_name.get(g.unit_keys[u][2], str(g.unit_keys[u][2]))
    return "quan hệ lạ" if r == "UNK_REL" else r


def merged_explanation(g, S, deltas, node_label, rel_name):
    """Trả về list ((đầu, quan hệ, đuôi), n, tổng Δ) sắp theo Δ giảm dần; đầu/đuôi là nhãn, ảnh gốc là ROOT."""
    key_of = lambda v: ROOT if v == g.root else node_label(g, v)
    groups = OrderedDict()
    for u in S:
        s, d = _endpoints(g, u)
        k = (key_of(s), _rel(g, u, rel_name), key_of(d))
        n, t = groups.get(k, (0, 0.0))
        groups[k] = (n + 1, t + float(deltas[u]))
    return sorted(((k, n, t) for k, (n, t) in groups.items()), key=lambda x: -x[2])


def _sentence(k):
    s, r, d = k
    return f"ảnh chứa {d}" if s == ROOT and r == "contains" else f"{s} {r} {d}"


def _layout(G, tries=30):
    """Spring layout (như cấp 2), chọn seed để không cạnh nào đi xuyên qua nút khác và các nút không quá sát nhau."""
    import numpy as np

    def penalty(pos):
        P = {k: np.asarray(v, float) for k, v in pos.items()}
        span = max(np.ptp([p[0] for p in P.values()]), np.ptp([p[1] for p in P.values()]), 1e-9)
        bad = 0.0
        for a, b in G.edges:
            A, B = P[a], P[b]
            AB = B - A
            for c, C in P.items():
                if c in (a, b):
                    continue
                t = np.clip(np.dot(C - A, AB) / max(np.dot(AB, AB), 1e-12), 0, 1)
                dist = np.linalg.norm(A + t * AB - C) / span
                bad += max(0.0, 0.12 - dist) * 10
        keys = list(P)
        for i in range(len(keys)):
            for j in range(i + 1, len(keys)):
                bad += max(0.0, 0.15 - np.linalg.norm(P[keys[i]] - P[keys[j]]) / span) * 5
        return bad

    best, best_pen = None, float("inf")
    for seed in range(tries):
        pos = nx.spring_layout(G, seed=seed, k=1.2)
        pen = penalty(pos)
        if pen < best_pen:
            best, best_pen = pos, pen
        if pen == 0:
            break
    return best


def draw_graph(ax, items, k_star):
    ax.axis("off")
    if not items:
        ax.text(.5, .5, "Không có quan hệ nào\nlàm tăng điểm đồ thị", ha="center", va="center", fontsize=FS["title"],
                transform=ax.transAxes)
        return
    G = nx.DiGraph()
    G.add_node(ROOT)
    size, mult, loops = {}, {}, {}
    # Δ ghi trên cạnh theo một thang 10^expo chung; độ dày cạnh tỉ lệ với Δ.
    expo = int(math.floor(math.log10(max(t for _, _, t in items))))
    for (s, r, d), n, t in items:
        for x in (s, d):
            G.add_node(x)
            if x != ROOT:
                size[x] = size.get(x, 0.0) + t
        if s == ROOT:
            mult[d] = max(mult.get(d, 1), n)
        lab = (("chứa" if r == "contains" else r) + (f" ×{n}" if n > 1 and s != ROOT else "")
               + f" ({t / 10 ** expo:.3g})")
        if s == d:                # hai vật cùng tên có quan hệ với nhau: ghi ngay dưới tên nút
            loops.setdefault(s, []).append(lab)
            continue
        if G.has_edge(s, d):      # nhiều quan hệ trên cùng một cặp: ghi đủ, không để nhãn sau đè nhãn trước
            G.edges[s, d]["label"] += " / " + lab
            G.edges[s, d]["delta"] += t
        else:
            G.add_edge(s, d, label=lab, delta=t)
    for e in G.edges:
        G.edges[e]["label"] = textwrap.shorten(G.edges[e]["label"], 40, placeholder="...")
    vmax = max(size.values()) if size else 1.0
    dmax = max(G.edges[e]["delta"] for e in G.edges) if G.edges else 1.0
    pos = _layout(G)
    nodes = list(G)
    nsize = {x: 900 if x == ROOT else 250 + 900 * max(size.get(x, 0), 0) / vmax for x in nodes}
    nx.draw_networkx_nodes(G, pos, ax=ax, nodelist=nodes, node_color=[C_IMG if x == ROOT else C_OTHER for x in nodes],
                           node_size=[nsize[x] for x in nodes])
    # cặp có quan hệ theo cả hai chiều: uốn cong để hai cạnh và hai nhãn tách nhau
    two_way = [e for e in G.edges if G.has_edge(e[1], e[0])]
    one_way = [e for e in G.edges if e not in two_way]
    for elist, cs in ((one_way, "arc3"), (two_way, "arc3,rad=0.25")):
        if not elist:
            continue
        nx.draw_networkx_edges(G, pos, ax=ax, edgelist=elist, edge_color=C_EDGE, arrows=True, arrowsize=12,
                               width=[1.0 + 3.5 * G.edges[e]["delta"] / dmax for e in elist],
                               nodelist=nodes, node_size=[nsize[x] for x in nodes], connectionstyle=cs)
        nx.draw_networkx_edge_labels(G, pos, ax=ax, edge_labels={e: G.edges[e]["label"] for e in elist},
                                     font_size=FS["edge"], connectionstyle=cs,
                                     bbox=dict(fc="white", ec="none", alpha=0.85, pad=0.5))
    labels = {x: x if x == ROOT else textwrap.shorten(x, 22, placeholder="...") + (f" ×{mult[x]}" if mult.get(x, 1) > 1 else "")
              for x in nodes}
    for x, labs in loops.items():
        labels[x] += "\n(" + ", ".join(labs) + f" {textwrap.shorten(x, 14, placeholder='...')})"
    nx.draw_networkx_labels(G, pos, ax=ax, labels=labels, font_size=FS["node"])
    xs = [p[0] for p in pos.values()]; ys = [p[1] for p in pos.values()]
    ax.set_xlim(min(xs) - 0.45, max(xs) + 0.45); ax.set_ylim(min(ys) - 0.22, max(ys) + 0.22)
    ax.set_title(f"Giải thích S (K* = {k_star}); số trong ngoặc: Δ điểm đồ thị ×10$^{{{expo}}}$",
                 fontsize=FS["title"])


def _show_image(ax, image, title):
    if image is not None:
        ax.imshow(image)
    else:
        ax.text(.5, .5, "không tải được ảnh", ha="center", va="center", transform=ax.transAxes)
    ax.set_title(title, fontsize=FS["title"])
    ax.axis("off")


def case_figure_items(images, items, k_star, suptitle):
    """Như case_figure nhưng nhận thẳng items của merged_explanation (dùng khi vẽ lại từ V6_cases.md)."""
    n = len(images)
    fig, axes = plt.subplots(1, n + 1, figsize=(FIG_W / 2.6 * (n + 1.6), FIG_H),
                             gridspec_kw={"width_ratios": [1.0] * n + [1.6]})
    for ax, (title, img) in zip(axes, images):
        _show_image(ax, img, title)
    draw_graph(axes[n], items, k_star)
    fig.suptitle(suptitle, fontsize=FS["sup"])
    fig.tight_layout()
    return fig


def case_figure(images, g, S, deltas, node_label, rel_name, k_star, suptitle):
    """images: list (tiêu đề, ảnh PIL hoặc None); giải thích (g, S, deltas) là của ảnh đầu tiên, như cấp 2."""
    return case_figure_items(images, merged_explanation(g, S, deltas, node_label, rel_name), k_star, suptitle)


def plot_explained_row(axes, g, S, deltas, image, header, node_label, rel_name, k_star):
    """Một hàng (ảnh, đồ thị) cho demo nhiều kết quả."""
    _show_image(axes[0], image, header)
    items = merged_explanation(g, S, deltas, node_label, rel_name)
    draw_graph(axes[1], items, k_star)


def new_figure(n_rows):
    fig, axes = plt.subplots(n_rows, 2, figsize=(FIG_W, FIG_H * n_rows), squeeze=False,
                             gridspec_kw={"width_ratios": [1.0, 1.6]})
    return fig, axes


def unit_sentence(g, u, node_label, rel_name):
    s, d = _endpoints(g, u)
    r = _rel(g, u, rel_name)
    if r == "contains":
        return f"ảnh chứa {node_label(g, d if s == g.root else s)}"
    return f"{node_label(g, s)} {r} {node_label(g, d)}"


def vg_image_urls(image_id):
    """Đường dẫn ảnh Visual Genome công khai theo số hiệu ảnh (hai thư mục của bộ dữ liệu gốc)."""
    num = str(image_id).split(":")[-1]
    return [f"https://cs.stanford.edu/people/rak248/VG_100K/{num}.jpg",
            f"https://cs.stanford.edu/people/rak248/VG_100K_2/{num}.jpg"]

In [ ]:
import requests, io
from PIL import Image

_IMG_CACHE = {}
def fetch_image(image_id):
    # Ưu tiên URL trong images.parquet, sau đó thử hai thư mục ảnh công khai của Visual Genome.
    key = str(image_id)
    if key in _IMG_CACHE:
        return _IMG_CACHE[key]
    urls = ([IMG_URL[key]] if IMG_URL.get(key) else []) + vg_image_urls(key)
    img = None
    for url in urls:
        try:
            r = requests.get(url, timeout=15); r.raise_for_status()
            img = Image.open(io.BytesIO(r.content)).convert('RGB'); break
        except Exception:
            continue
    _IMG_CACHE[key] = img
    return img

## Giao diện dùng chung (nhúng nguyên văn `src/multimodal_xai/demo_web.py`)

In [ ]:
# Giao diện web demo dùng chung cho ba cấp độ (Visual Genome, DB15K / FB15K-237-IMG, ViHeritage-MMKG).
#
# Mỗi notebook demo chỉ tạo dữ liệu (header + danh sách kết quả); trang HTML, đồ thị lực kiểu Obsidian (force-graph),
# màu sắc và cách đọc giống nhau ở cả ba cấp. Màu theo báo cáo (report/figures/scripts/make_figures.py):
#   xanh ngọc = thực thể neo, cam = ảnh / thực thể của ảnh trả về, xanh dương = phần tử của tập giải thích S,
#   xanh nhạt = đường nối về neo (chỉ để đọc, không đo), xám nét đứt = quan hệ được hỏi, xám = nút khác.
# Màn hình chính chỉ hiện phần cần cho demo; số liệu phụ nằm trong mục "Chi tiết" (mặc định đóng).
# File này được nhúng nguyên văn vào notebook demo cấp 1 và cấp 3; notebook cấp 2 import từ package.
#
# Dữ liệu một kết quả (item):
#   rank, title, correct (True / False / None), img (data URL), graph = {nodes, links} (dùng node(), link()),
#   empty (chữ hiện khi không có đồ thị), why = [(nhãn, nội dung)] (kiểm chứng chính, tối đa 3 dòng),
#   more = [(nhãn, nội dung)] (chi tiết), bars = [{label, w, text}] (chi tiết), note (chi tiết),
#   row = {clip_rank, removed} (bảng xếp hạng),
#   story = một câu trên đồ thị trả lời "Vì sao trả về X?" (bằng chứng mạnh nhất, đường đi về neo),
#   removal = câu hiện khi bấm "Bỏ giải thích S" (hạng trước, sau; so với bỏ ngẫu nhiên).
import base64
import html as _html
import io
import json
import os
import urllib.request
from pathlib import Path

PALETTE = {"anchor": "#1baf7a", "cand": "#eb6834", "other": "#d9d8d4",
           "fact": "#2a78d6", "path": "#9cc0ef", "query": "#8f8e8a"}
PAGE_HEIGHT = 720

_FG_URLS = ["https://cdn.jsdelivr.net/npm/force-graph@1.52.0/dist/force-graph.min.js",
            "https://unpkg.com/force-graph@1.52.0/dist/force-graph.min.js"]
_FG = {}


def force_graph_js():
    """Nhúng thẳng force-graph vào trang (iframe srcdoc không tải được script ngoài ở một số trình duyệt)."""
    if "js" not in _FG:
        _FG["js"] = None
        local = os.environ.get("XAI_FORCEGRAPH_JS")
        if local and Path(local).exists():
            _FG["js"] = Path(local).read_text(encoding="utf-8")
        else:
            for u in _FG_URLS:
                try:
                    _FG["js"] = urllib.request.urlopen(u, timeout=30).read().decode("utf-8")
                    break
                except Exception:
                    continue
    return f"<script>{_FG['js']}</script>" if _FG["js"] else f'<script src="{_FG_URLS[0]}"></script>'


def image_data_url(img, size=640, quality=85):
    """PIL.Image hoặc đường dẫn → data URL JPEG; None nếu không mở được."""
    try:
        from PIL import Image
        if img is None:
            return None
        if not hasattr(img, "convert"):
            img = Image.open(img)
        img = img.convert("RGB")
        img.thumbnail((size, size))
        b = io.BytesIO()
        img.save(b, format="JPEG", quality=quality)
        return "data:image/jpeg;base64," + base64.b64encode(b.getvalue()).decode()
    except Exception:
        return None


def rank_text(r):
    if r is None:
        return "không có"
    r = float(r)
    return f"{int(r)}" if r == int(r) else f"{r:.1f}"


def _move(r0, x):
    if x is None:
        return "không đo"
    if abs(float(x) - r0) < 0.05:
        return f"vẫn hạng {r0}"
    return f"{'xuống' if x > r0 else 'lên'} hạng {rank_text(x)}"


def removal_sentence(n, unit, r0, after, rand=None):
    """Câu hiện khi bấm "Bỏ giải thích S": hạng sau khi bỏ S, so với bỏ cùng số phần tử ngẫu nhiên (trung bình)."""
    s = f"Bỏ {n} {unit} giải thích: ảnh {_move(r0, after)}"
    if rand is not None:
        s += f" · bỏ {n} {unit} ngẫu nhiên: {_move(r0, rand)}" + (" (trung bình)" if float(rand) != int(rand) else "")
    return s


def node(nid, name, kind, val=2.0, tip=None, result=False):
    """result=True: nút này cũng là thực thể của ảnh trả về (ví dụ mô hình trả về chính thực thể neo), vẽ bằng bức ảnh."""
    return dict(id=nid, name=str(name), kind=kind, val=float(val), tip=str(tip or name), result=bool(result))


def link(source, target, rel, kind, delta=0.0, label=None, tip=None):
    return dict(source=source, target=target, rel=str(rel), kind=kind, delta=float(delta),
                label=str(rel if label is None else label), tip=str(tip or rel))


DEFAULT_LEGEND = {"anchor": "neo", "cand": "ảnh trả về", "other": "nút khác",
                  "fact": "giải thích S (dày theo Δ)", "path": "nối về neo", "query": "quan hệ được hỏi"}

_CSS = r"""
:root{--bg:#1e1e1e;--panel:#252525;--panel2:#2b2b2b;--line:#353535;--text:#e3e3e0;--muted:#9a9a96;--accent:#2a78d6;--good:#1baf7a;--bad:#eb6834}
body.light{--bg:#ffffff;--panel:#f6f6f4;--panel2:#ececea;--line:#dcdbd7;--text:#1f1f1d;--muted:#6b6a66}
*{box-sizing:border-box}html,body{height:100%}
body{margin:0;background:var(--bg);color:var(--text);font:14px/1.45 system-ui,-apple-system,'Segoe UI',Roboto,sans-serif;display:flex;flex-direction:column;overflow:hidden}
.hdr{padding:10px 14px 8px;border-bottom:1px solid var(--line);display:flex;flex-wrap:wrap;align-items:baseline;gap:4px 18px}
.q{font-size:16px;font-weight:600}.meta{font-size:13.5px;color:var(--muted)}.meta b{color:var(--text);font-weight:600}
.tg{font-size:14px;font-weight:600;margin-left:auto}
.tabs{display:flex;gap:2px;padding:2px 10px 0;border-bottom:1px solid var(--line)}
.tab{background:none;border:0;color:var(--muted);padding:7px 14px;font:inherit;font-weight:600;cursor:pointer;border-bottom:2px solid transparent}
.tab.on{color:var(--text);border-bottom-color:var(--accent)}
.pane{display:none;flex:1;min-height:0;padding:10px 12px}.pane.on{display:block}
.gv{display:grid;grid-template-columns:minmax(0,1fr) minmax(0,1.4fr);gap:12px;height:100%}
.viewer{display:flex;flex-direction:column;min-width:0;min-height:0;overflow:auto;padding-right:4px}
.vhead{margin-bottom:6px;font-size:15px}
.ok{color:var(--good);font-weight:700}.no{color:var(--bad);font-weight:700}
.stage{position:relative;background:var(--panel);border-radius:8px;display:flex;align-items:center;justify-content:center;height:290px;flex:none;overflow:hidden}
.stage img{max-width:100%;max-height:290px;display:block}.ph{color:var(--muted);font-size:13px}
.nav{position:absolute;top:50%;transform:translateY(-50%);background:var(--panel2);border:1px solid var(--line);color:var(--text);width:34px;height:34px;border-radius:50%;cursor:pointer;font-size:18px;line-height:30px;z-index:3;opacity:.9}
.nav:hover{border-color:var(--accent)}.nav.prev{left:8px}.nav.next{right:8px}
.thumbs{display:flex;gap:6px;margin-top:8px;flex:none}
.thumbs div{flex:1;height:48px;border-radius:5px;background:var(--panel2) center/cover no-repeat;cursor:pointer;border:2px solid transparent;opacity:.6;position:relative}
.thumbs div.on{border-color:var(--accent);opacity:1}
.thumbs span{position:absolute;left:3px;top:1px;font-size:11px;font-weight:700;color:#fff;text-shadow:0 0 3px #000}
.box{margin-top:10px;background:var(--panel);border:1px solid var(--line);border-radius:8px;padding:8px 12px;flex:none}
.box h4{margin:0 0 4px;font-size:12px;color:var(--muted);font-weight:600}
.why{font-size:14.5px;margin:3px 0}.why .l{color:var(--muted)}.why .v{font-weight:700}
details.more{margin-top:8px;font-size:12.5px;color:var(--muted);flex:none}details.more summary{cursor:pointer}
.kv{display:grid;grid-template-columns:auto 1fr;gap:2px 10px;margin-top:4px}.kv span:nth-child(even){color:var(--text)}
.bar{display:flex;align-items:center;gap:8px;margin:3px 0}.bar .n{width:96px}
.bar i{display:block;height:7px;background:var(--accent);border-radius:4px;flex:none}.bar span:last-child{color:var(--text)}
.gwrap{position:relative;border-radius:8px;overflow:hidden;background:var(--bg);border:1px solid var(--line);min-width:0}
#graph{position:absolute;inset:0}
.empty{position:absolute;inset:0;display:none;align-items:center;justify-content:center;text-align:center;color:var(--muted);padding:28px;font-size:14px}
.gear{position:absolute;top:8px;right:8px;z-index:4;background:var(--panel2);border:1px solid var(--line);color:var(--text);border-radius:6px;padding:4px 9px;cursor:pointer;font:inherit;font-size:13px}
.panel{position:absolute;top:42px;right:8px;width:260px;z-index:4;background:var(--panel);border:1px solid var(--line);border-radius:8px;padding:10px 12px;display:none;font-size:12.5px}
.panel.on{display:block}
.panel label{display:flex;justify-content:space-between;align-items:center;margin:5px 0;gap:8px}
.panel input[type=range]{width:100px;accent-color:var(--accent)}.panel input[type=checkbox]{accent-color:var(--accent)}
.panel button{width:100%;margin-top:8px;background:var(--panel2);border:1px solid var(--line);color:var(--text);border-radius:5px;padding:5px;cursor:pointer;font:inherit}
.story{position:absolute;left:8px;top:8px;right:150px;z-index:3;background:var(--panel);opacity:.96;border:1px solid var(--line);border-radius:8px;padding:6px 11px;pointer-events:none}
.story .t{font-size:15.5px;font-weight:700}.story .s{font-size:13.5px;margin-top:2px}
.rmS{position:absolute;top:8px;right:48px;z-index:4;background:var(--accent);border:0;color:#fff;border-radius:6px;padding:5px 11px;cursor:pointer;font:inherit;font-size:13px;font-weight:600}
.rmS.on{background:var(--panel2);color:var(--text);border:1px solid var(--line)}
.banner{position:absolute;left:50%;transform:translateX(-50%);top:78px;z-index:3;background:var(--panel);border:2px solid var(--bad);border-radius:8px;padding:7px 14px;font-size:14.5px;font-weight:600;display:none;text-align:center;max-width:90%}
.legend{position:absolute;left:8px;bottom:8px;z-index:3;font-size:12.5px;color:var(--text);background:var(--panel);opacity:.95;padding:4px 10px;border-radius:6px;border:1px solid var(--line);pointer-events:none;display:flex;flex-wrap:wrap;gap:2px 14px;max-width:calc(100% - 16px)}
.dot{display:inline-block;width:10px;height:10px;border-radius:50%;margin:0 5px -1px 0}
.ln{display:inline-block;width:20px;height:0;margin:0 5px 3px 0;border-top:3px solid}
table{width:100%;border-collapse:collapse;font-size:14px}th{text-align:left;color:var(--muted);font-weight:600;font-size:13px;padding:7px 10px;border-bottom:1px solid var(--line)}
td{padding:7px 10px;border-bottom:1px solid var(--line)}tbody tr{cursor:pointer}tbody tr:hover{background:var(--panel)}
td .th{width:84px;height:56px;border-radius:4px;background:var(--panel2) center/cover no-repeat}
.up{color:var(--good)}.down{color:var(--bad)}
#p-rank{overflow:auto}
.float-tooltip-kap{background:var(--panel)!important;color:var(--text)!important;border:1px solid var(--line);font-size:12.5px!important;padding:4px 8px!important;border-radius:5px}
"""

_BODY = r"""
<div class="hdr" id="hdr"></div>
<div class="tabs"><button class="tab on" data-t="graph">Đồ thị giải thích</button><button class="tab" data-t="rank">Bảng xếp hạng</button></div>
<div class="pane on" id="p-graph"><div class="gv">
 <div class="viewer">
  <div class="vhead" id="vh"></div>
  <div class="stage"><button class="nav prev" title="Ảnh trước (←)">‹</button><div id="stage-img"></div><button class="nav next" title="Ảnh sau (→)">›</button></div>
  <div class="thumbs" id="thumbs"></div>
  <div id="info"></div>
 </div>
 <div class="gwrap"><div id="graph"></div><div class="empty"></div><div class="story" id="story"></div>
  <button class="rmS" id="rmS"></button><div class="banner" id="banner"></div><button class="gear">⚙</button>
  <div class="panel">
   <label>Nền sáng<input type="checkbox" data-k="light"></label>
   <label>Nhãn quan hệ<input type="checkbox" data-k="labels" checked></label><label>Tên nút<input type="checkbox" data-k="names" checked></label>
   <label>Mũi tên<input type="checkbox" data-k="arrows" checked></label><label>Đường nối về neo<input type="checkbox" data-k="paths" checked></label>
   <label>Kích thước nút<input type="range" data-k="size" min="0.5" max="2" step="0.1" value="1"></label><label>Độ dày cạnh<input type="range" data-k="width" min="0.3" max="3" step="0.1" value="1"></label>
   <label>Lực đẩy<input type="range" data-k="repel" min="20" max="800" step="10" value="160"></label>
   <label>Độ dài cạnh<input type="range" data-k="dist" min="20" max="260" step="5" value="95"></label><button id="fit">Căn vừa khung</button></div>
  <div class="legend" id="legend"></div></div></div></div>
<div class="pane" id="p-rank"><table><thead><tr id="rank-head"></tr></thead><tbody id="rows"></tbody></table></div>
"""

_JS = r"""
const $=s=>document.querySelector(s),$$=s=>[...document.querySelectorAll(s)];
const esc=t=>String(t??'').replace(/[&<>"]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));
const P=D.palette;
const THEME={dark:{bg:'#1e1e1e',text:'#e3e3e0',muted:'#9a9a96',label:'#9cc0ef',qlabel:'#b8b7b2'},
             light:{bg:'#ffffff',text:'#1f1f1d',muted:'#6b6a66',label:'#1c5fae',qlabel:'#52514e'}};
const S={light:false,labels:true,names:true,arrows:true,paths:true,size:1,width:1,repel:160,dist:95};
let cur=0,hover=null,nb={},fg=null;const gel=$('#graph');
const T=()=>THEME[S.light?'light':'dark'];
function rgba(hex,a){const n=parseInt(hex.slice(1),16);return `rgba(${n>>16&255},${n>>8&255},${n&255},${a})`;}
function tab(t){$$('.tab').forEach(b=>b.classList.toggle('on',b.dataset.t===t));$$('.pane').forEach(p=>p.classList.toggle('on',p.id==='p-'+t));
  if(t==='graph'&&fg)fg.width(gel.clientWidth).height(gel.clientHeight);}
$$('.tab').forEach(b=>b.onclick=()=>tab(b.dataset.t));
// ---------- header: câu truy vấn, neo / quan hệ, hạng ảnh đúng ----------
(function(){const h=D.header;let s='<span class="q">“'+esc(h.query)+'”</span>';
  if(h.chips&&h.chips.length)s+='<span class="meta">'+h.chips.map(c=>esc(c[0])+': <b>'+esc(c[1])+'</b>').join(' · ')+'</span>';
  if(h.target)s+='<span class="tg">'+esc(h.target)+'</span>';$('#hdr').innerHTML=s;})();
// ---------- đồ thị ----------
const rad=n=>Math.sqrt(n.val)*4*S.size;
function mid(l){const s=l.source,t=l.target,c=l.__controlPoints;
  if(c&&c.length===2)return{x:.25*s.x+.5*c[0]+.25*t.x,y:.25*s.y+.5*c[1]+.25*t.y};
  if(c&&c.length===4)return{x:.125*s.x+.375*c[0]+.375*c[2]+.125*t.x,y:.125*s.y+.375*c[1]+.375*c[3]+.125*t.y};
  return{x:(s.x+t.x)/2,y:(s.y+t.y)/2};}
function active(l){return hover===null||l.source.id===hover||l.target.id===hover;}
const gone=l=>S.removed&&l.kind==='fact';
function lcol(l){if(gone(l))return rgba(P.query,.35);const c=P[l.kind]||P.fact,a=active(l);return rgba(c,l.kind==='path'?(a?.85:.15):(a?.95:.15));}
const IMG={};function imgOf(src){if(!src)return null;if(!IMG[src]){const im=new Image();im.src=src;IMG[src]=im;}return IMG[src];}
function collide(){let nodes=[];const f=alpha=>{for(let i=0;i<nodes.length;i++)for(let j=i+1;j<nodes.length;j++){
    const a=nodes[i],b=nodes[j];let dx=b.x-a.x,dy=b.y-a.y,d=Math.hypot(dx,dy)||1e-3;const m=rad(a)+rad(b)+34;
    if(d<m){const k=(m-d)/d*.5*Math.min(1,alpha*4);dx*=k;dy*=k;
      if(a.fx===undefined){a.x-=dx;a.y-=dy;}if(b.fx===undefined){b.x+=dx;b.y+=dy;}}}};f.initialize=n=>{nodes=n;};return f;}
function initGraph(){ if(fg||!window.ForceGraph)return;
  fg=ForceGraph()(gel).backgroundColor(T().bg).autoPauseRedraw(false).nodeId('id').nodeVal(n=>n.val).nodeRelSize(4)
   .nodeLabel(n=>esc(n.tip)).linkLabel(l=>esc(l.tip)).linkVisibility(l=>l.kind!=='path'||S.paths)
   .linkColor(lcol).linkCurvature(l=>l.curv||0)
   .linkWidth(l=>(gone(l)?1:l.kind==='fact'?1.4+4*Math.min(1,Math.abs(l.delta)/(D.items[cur].dmax||1)):(l.kind==='query'?1.4:1))*S.width)
   .linkLineDash(l=>gone(l)?[3,5]:l.kind==='query'?[5,4]:(l.kind==='path'?[2,3]:null))
   .linkDirectionalArrowLength(l=>S.arrows&&!gone(l)?(l.kind==='fact'?6:4.5):0).linkDirectionalArrowRelPos(1).linkDirectionalArrowColor(lcol)
   .linkCanvasObjectMode(()=>'after').linkCanvasObject((l,ctx,s)=>{
      if(!S.labels||s<0.35||!l.label||(l.kind==='path'&&!S.paths))return;
      const m=mid(l),f=Math.max(11.5/s,2.5);ctx.font=f+'px system-ui,sans-serif';ctx.textAlign='center';ctx.textBaseline='middle';
      const w=ctx.measureText(l.label).width;ctx.globalAlpha=gone(l)?.3:active(l)?1:.25;
      ctx.fillStyle=T().bg;ctx.fillRect(m.x-w/2-3/s,m.y-f/2-1.5/s,w+6/s,f+3/s);
      ctx.fillStyle=l.kind==='query'?T().qlabel:(l.kind==='path'?T().muted:T().label);ctx.fillText(l.label,m.x,m.y);ctx.globalAlpha=1;})
   .nodeCanvasObject((n,ctx,s)=>{const r=rad(n),act=hover===null||(nb[hover]&&nb[hover].has(n.id));
      ctx.globalAlpha=act?1:.25;ctx.beginPath();ctx.arc(n.x,n.y,r+2/s,0,2*Math.PI);ctx.fillStyle=T().bg;ctx.fill();
      ctx.beginPath();ctx.arc(n.x,n.y,r,0,2*Math.PI);ctx.fillStyle=P[n.kind]||P.other;ctx.fill();
      const im=n.img&&imgOf(n.img);
      if(im&&im.complete&&im.naturalWidth){ctx.save();ctx.beginPath();ctx.arc(n.x,n.y,r,0,2*Math.PI);ctx.clip();
        const k=Math.max(2*r/im.naturalWidth,2*r/im.naturalHeight);ctx.drawImage(im,n.x-im.naturalWidth*k/2,n.y-im.naturalHeight*k/2,im.naturalWidth*k,im.naturalHeight*k);
        ctx.restore();ctx.beginPath();ctx.arc(n.x,n.y,r,0,2*Math.PI);ctx.lineWidth=4/s;ctx.strokeStyle=P.cand;ctx.stroke();}
      if(n.id===hover){ctx.lineWidth=2.5/s;ctx.strokeStyle=T().text;ctx.stroke();}
      ctx.globalAlpha=1;if(!S.names&&n.id!==hover)return;
      const f=Math.max(12.5/s,3);ctx.font=(n.kind!=='other'?'600 ':'')+f+'px system-ui,sans-serif';ctx.textAlign='center';ctx.textBaseline='top';
      ctx.lineWidth=3.5/s;ctx.strokeStyle=T().bg;ctx.lineJoin='round';ctx.strokeText(n.name,n.x,n.y+r+3/s);
      ctx.fillStyle=act?T().text:rgba(T().text,.3);ctx.fillText(n.name,n.x,n.y+r+3/s);
      const rk=D.roles.cand+' '+D.items[cur].rank;
      const role=n.role||(n.kind==='anchor'?(n.result?D.roles.anchor+', cũng là '+rk:D.roles.anchor):n.kind==='cand'?rk:'');
      if(role){const f2=Math.max(11/s,2.5);ctx.font=f2+'px system-ui,sans-serif';ctx.strokeText(role,n.x,n.y+r+5/s+f);
        ctx.fillStyle=n.kind==='anchor'?P.anchor:P.cand;ctx.fillText(role,n.x,n.y+r+5/s+f);}})
   .nodePointerAreaPaint((n,c,ctx)=>{ctx.fillStyle=c;ctx.beginPath();ctx.arc(n.x,n.y,rad(n)+4,0,2*Math.PI);ctx.fill();})
   .onNodeHover(n=>{hover=n?n.id:null;gel.style.cursor=n?'pointer':null;})
   .cooldownTicks(160).onEngineStop(()=>fg.zoomToFit(400,90));
  forces();}
function forces(){if(!fg)return;fg.d3Force('charge').strength(-S.repel);fg.d3Force('collide',collide());
  fg.d3Force('link').distance(l=>l.kind==='fact'?S.dist:S.dist*1.25);fg.nodeRelSize(4*S.size);fg.d3ReheatSimulation();}
function seedLayout(g,fix){// bố cục ban đầu cố định (nan quạt quanh nút neo hoặc nút ảnh) để mỗi lần mở đều ra cùng một hình
  const others=g.nodes.filter(n=>n!==fix),cands=others.filter(n=>n.kind==='cand'),rest=others.filter(n=>n.kind!=='cand'),R=S.dist*1.7;
  cands.forEach((n,i)=>{n.x=(i-(cands.length-1)/2)*S.dist;n.y=S.dist*1.2;});
  rest.forEach((n,i)=>{const a=cands.length?(-Math.PI*0.1-Math.PI*0.8*(i+.5)/rest.length):(2*Math.PI*i/rest.length-Math.PI/2);
    n.x=R*Math.cos(a)*(cands.length?1.25:1);n.y=R*Math.sin(a);});}
function curvatures(g){const grp={};
  g.links.forEach(l=>{const a=String(l.source),b=String(l.target),k=a<b?a+'|'+b:b+'|'+a;(grp[k]=grp[k]||[]).push(l);});
  Object.values(grp).forEach(arr=>arr.forEach((l,i)=>{
    if(String(l.source)===String(l.target)){l.curv=1+.4*i;return;}
    if(arr.length===1){l.curv=0;return;}
    let c=(i-(arr.length-1)/2)*.8;if(String(l.source)>String(l.target))c=-c;l.curv=c;}));}
function legend(it){const L=D.legend,kinds=new Set(it.graph.nodes.map(n=>n.kind)),lk=new Set(it.graph.links.map(l=>l.kind));const r=[];
  ['anchor','cand','other'].forEach(k=>{if(kinds.has(k)&&L[k])r.push('<span><span class="dot" style="background:'+P[k]+'"></span>'+esc(L[k])+'</span>');});
  if(lk.has('fact')&&L.fact)r.push('<span><span class="ln" style="border-color:'+P.fact+'"></span>'+esc(L.fact)+'</span>');
  if(lk.has('path')&&L.path)r.push('<span><span class="ln" style="border-color:'+P.path+';border-top-style:dotted"></span>'+esc(L.path)+'</span>');
  if(lk.has('query')&&L.query)r.push('<span><span class="ln" style="border-color:'+P.query+';border-top-style:dashed"></span>'+esc(L.query)+'</span>');
  $('#legend').innerHTML=r.join('');$('#legend').style.display=r.length?'flex':'none';}
// ---------- cột trái: chỉ phần cần cho demo, số liệu phụ trong "Chi tiết" ----------
function badge(it){return it.correct===true?' <span class="ok">✓ ảnh đúng</span>':(it.correct===false?' <span class="no">✗ chưa đúng</span>':'');}
function info(it){let s='';
  if(it.why&&it.why.length)s+='<div class="box"><h4>'+esc(D.check_title)+'</h4>'+it.why.map(x=>'<div class="why"><span class="l">'+esc(x[0])+':</span> <span class="v">'+esc(x[1])+'</span></div>').join('')+'</div>';
  let m='';if(it.more&&it.more.length)m+='<div class="kv">'+it.more.map(x=>'<span>'+esc(x[0])+'</span><span>'+esc(x[1])+'</span>').join('')+'</div>';
  if(it.bars&&it.bars.length)m+='<div style="margin-top:6px">'+esc(D.bar_title)+'</div>'+it.bars.map(b=>'<div class="bar"><span class="n">'+esc(b.label)+'</span><i style="width:'+Math.max(3,Math.round(90*b.w))+'px"></i><span>'+esc(b.text)+'</span></div>').join('');
  if(it.note)m+='<div style="margin-top:6px">'+esc(it.note)+'</div>';
  if(m)s+='<details class="more"><summary>Chi tiết</summary>'+m+'</details>';return s;}
function show(i){cur=(i+D.items.length)%D.items.length;const it=D.items[cur];hover=null;
  $('#vh').innerHTML='<b>#'+it.rank+' · '+esc(it.title)+'</b>'+badge(it);
  $('#stage-img').innerHTML=it.img?'<img src="'+it.img+'" alt="'+esc(it.title)+'">':'<span class="ph">không mở được ảnh</span>';
  $('#info').innerHTML=info(it);$$('.thumbs div').forEach((t,k)=>t.classList.toggle('on',k===cur));
  const has=it.graph.links.length>0;$('.empty').style.display=has?'none':'flex';$('.empty').textContent=it.empty||'Không có phần tử đồ thị nào để vẽ cho ảnh này.';
  $('#story').innerHTML=has?'<div class="t">'+esc(D.story_title.replace('{x}',it.title))+'</div>'+(it.story?'<div class="s">'+esc(it.story)+'</div>':''):'';
  $('#story').style.display=has?'block':'none';
  S.removed=false;$('#banner').style.display='none';const rb=$('#rmS');rb.classList.remove('on');rb.textContent=D.remove_label;
  rb.style.display=has&&it.removal?'block':'none';
  legend(it);initGraph();if(!fg)return;
  const g=JSON.parse(JSON.stringify(it.graph));nb={};const fix=g.nodes.find(n=>n.kind==='anchor')||g.nodes.find(n=>n.kind==='cand');
  g.nodes.forEach(n=>{nb[n.id]=new Set([n.id]);if((n.kind==='cand'||n.result)&&it.img){n.img=it.img;n.val=30;}if(fix&&n.id===fix.id){n.fx=0;n.fy=0;n.x=0;n.y=0;}});seedLayout(g,fix);
  g.links.forEach(l=>{if(nb[l.source])nb[l.source].add(l.target);if(nb[l.target])nb[l.target].add(l.source);});curvatures(g);
  fg.width(gel.clientWidth).height(gel.clientHeight).graphData(has?g:{nodes:[],links:[]});
  const c0=cur;[60,700,1800].forEach(t=>setTimeout(()=>{if(fg&&cur===c0){fg.width(gel.clientWidth).height(gel.clientHeight);fg.zoomToFit(0,90);}},t));}
$('#thumbs').innerHTML=D.items.map((it,k)=>'<div title="'+esc(it.title)+'" style="background-image:url('+(it.img||'')+')" onclick="show('+k+')"><span>#'+it.rank+(it.correct?' ✓':'')+'</span></div>').join('');
$('.prev').onclick=()=>show(cur-1);$('.next').onclick=()=>show(cur+1);
document.addEventListener('keydown',e=>{if(!$('#p-graph').classList.contains('on'))return;if(e.key==='ArrowLeft')show(cur-1);if(e.key==='ArrowRight')show(cur+1);});
$('.gear').onclick=()=>$('.panel').classList.toggle('on');
$('#rmS').onclick=()=>{const it=D.items[cur];S.removed=!S.removed;const rb=$('#rmS');rb.classList.toggle('on',S.removed);
  rb.textContent=S.removed?D.restore_label:D.remove_label;$('#banner').textContent=it.removal||'';$('#banner').style.display=S.removed?'block':'none';};
$$('.panel input').forEach(inp=>inp.oninput=()=>{const k=inp.dataset.k;S[k]=inp.type==='checkbox'?inp.checked:+inp.value;
  if(k==='light'){document.body.classList.toggle('light',S.light);if(fg)fg.backgroundColor(T().bg);}
  if(['repel','dist','size'].includes(k))forces();});
$('#fit').onclick=()=>fg&&fg.zoomToFit(400,90);
// ---------- bảng xếp hạng ----------
$('#rank-head').innerHTML=['Hạng','Ảnh','Kết quả','Ảnh đúng','Hạng nếu chỉ dùng CLIP',D.removed_col].map(x=>'<th>'+esc(x)+'</th>').join('');
$('#rows').innerHTML=D.items.map((it,k)=>{const r=it.row||{},cr=r.clip_rank,d=(cr??it.rank)-it.rank;
  return '<tr onclick="tab(\'graph\');show('+k+')"><td><b>#'+it.rank+'</b></td><td><div class="th" style="background-image:url('+(it.img||'')+')"></div></td><td>'+esc(it.title)+'</td><td>'+(it.correct===true?'<span class="ok">✓</span>':'')+'</td><td>'+(cr==null?'':'#'+esc(cr))+(d>0?' <span class="up">▲'+d+'</span>':d<0?' <span class="down">▼'+(-d)+'</span>':'')+'</td><td>'+esc(r.removed??'')+'</td></tr>';}).join('');
window.addEventListener('resize',()=>fg&&fg.width(gel.clientWidth).height(gel.clientHeight));
(function(){const h=String(location.hash||'');if(h.includes('light')){S.light=true;document.body.classList.add('light');$('.panel input[data-k=light]').checked=true;}
  const m=h.match(/i=(\d+)/);if(m)cur=+m[1];if(h.includes('rank'))tab('rank');if(h.includes('more'))setTimeout(()=>$$('details.more').forEach(d=>d.open=true),0);
  if(h.includes('rm'))setTimeout(()=>$('#rmS').click(),50);})();
show(cur);if(fg)fg.backgroundColor(T().bg);
"""


def _prepare(items):
    for it in items:
        g = it.setdefault("graph", {"nodes": [], "links": []})
        it["dmax"] = max([abs(l["delta"]) for l in g["links"] if l["kind"] == "fact"] + [1e-12])
    return items


def results_page(header, items, legend=None, check_title="Kiểm chứng: bỏ giải thích rồi chấm lại",
                 bar_title="Che từng phương thức rồi chấm lại:", removed_col="Bỏ S → hạng", title="Demo truy vấn có giải thích",
                 story_title="Vì sao trả về {x}?", remove_label="Bỏ giải thích S", restore_label="Khôi phục S",
                 roles=None):
    """header: dict(query, chips=[(nhãn, giá trị)], target). items: danh sách kết quả theo hạng (xem đầu file).
    Mỗi item có thêm: story (một câu nêu bằng chứng mạnh nhất), removal (câu hiện khi bấm "Bỏ giải thích S")."""
    data = dict(header=header, items=_prepare(items), palette=PALETTE, legend={**DEFAULT_LEGEND, **(legend or {})},
                check_title=check_title, bar_title=bar_title, removed_col=removed_col, story_title=story_title,
                remove_label=remove_label, restore_label=restore_label,
                roles={"anchor": "neo của truy vấn", "cand": "ảnh hạng", **(roles or {})})
    payload = json.dumps(data, ensure_ascii=False).replace("</", "<\\/")
    return (f"<!DOCTYPE html><html lang='vi'><head><meta charset='utf-8'><title>{_html.escape(title)}</title><style>{_CSS}</style>"
            f"{force_graph_js()}</head><body>{_BODY}<script>const D={payload};{_JS}</script></body></html>")


def results_iframe(header, items, height=PAGE_HEIGHT, **kw):
    page = results_page(header, items, **kw)
    return (f'<iframe srcdoc="{_html.escape(page, quote=True)}" style="width:100%;height:{height}px;border:0;'
            f'border-radius:8px;background:#1e1e1e"></iframe>')


def message_html(text):
    return f'<div style="padding:14px;color:#9a9a96;font:14px system-ui,sans-serif">{_html.escape(text)}</div>'


def launch(demo, in_colab, height=PAGE_HEIGHT + 160):
    """Colab: mặc định tạo link công khai (*.gradio.live, tự hết hạn) để mở trang ở tab riêng khi quay video.
    Đặt XAI_DEMO_SHARE=0 để chỉ hiện trong notebook. Tắt notebook sau khi quay để link ngừng hoạt động."""
    share = os.environ.get("XAI_DEMO_SHARE", "1" if in_colab else "0") == "1"
    demo.queue()
    if share:
        return demo.launch(share=True, inline=False, show_error=True)
    return demo.launch(share=False, inline=in_colab, height=height, show_error=True)

# Gom các hàm của giao diện vào W (giống `from multimodal_xai import demo_web as W` ở notebook cấp 2)
import types as _types
W = _types.SimpleNamespace(PALETTE=PALETTE, PAGE_HEIGHT=PAGE_HEIGHT, image_data_url=image_data_url, rank_text=rank_text, removal_sentence=removal_sentence, node=node, link=link, results_page=results_page, results_iframe=results_iframe, message_html=message_html, launch=launch)

## Web demo

In [ ]:
# Web demo cấp 1 (Visual Genome, CLIP + EdgeAwareGAT). Nhúng nguyên văn vào demo_cap1_visual_genome.ipynb, chạy sau các ô
# nạp dữ liệu, mô hình, lõi giải thích (vg_xai_core.py), chấm điểm, vg_xai_plot.py và demo_web.py.
# Giải thích = tập S gồm tối đa K* quan hệ của scene graph có Δ dương lớn nhất (Δ = điểm đồ thị giảm khi bỏ quan hệ đó),
# kiểm chứng bằng chính explain_and_measure của notebook 11: bỏ S, bỏ cùng số quan hệ ngẫu nhiên (10 lần), rồi xếp hạng lại.
import os
from collections import Counter

MAIN = f'seed_{MAIN_SEED}'
SPEC = MODELS[MAIN]
K_STAR = 12
if (OUT / '11_run_info.json').exists():
    K_STAR = int(json.loads((OUT / '11_run_info.json').read_text())['K_star'])
elif (OUT / 'V2_Kstar_val.csv').exists():
    _kd = pd.read_csv(OUT / 'V2_Kstar_val.csv'); K_STAR = int(_kd.loc[_kd.is_K_star, 'K'].iloc[0])
test_res = SPLIT['test']['res']
ROW_OF = {str(t): i for i, t in enumerate(test_res.text_id.astype(str))}

# ---------- 10 ca của báo cáo: chọn lại đúng luật V6 từ bảng hạng đã lưu (không chấm lại 20 000 truy vấn) ----------
DEMO_CASES = {}
_p = OUT / 'V6_case_pool_ranks.csv'
if _p.exists():
    rk = pd.read_csv(_p, dtype={'text_id': str, 'positive': str, 'top': str})
    def _pick(df, n, sort=None, seed=SELECTION_SEED):
        df = df.drop_duplicates('positive')
        df = df.sort_values(sort, ascending=False) if sort else df.sample(frac=1.0, random_state=seed)
        return df.head(n)
    succ = _pick(rk[(rk.r_clip > 1) & (rk.r_gat == 1) & (rk.r_no_edge > 1) & (rk.r_random > 1)].assign(gain=lambda d: d.r_clip - d.r_gat), 5, 'gain')
    harm = _pick(rk[(rk.r_clip == 1) & (rk.r_gat > 1) & ~rk.positive.isin(succ.positive)], 3)
    miss = _pick(rk[(rk.r_clip > 1) & (rk.r_gat.between(2, 5)) & ~rk.positive.isin(pd.concat([succ, harm]).positive)], 2)
    for kind, df in [('thành công', succ), ('thất bại, đồ thị làm hại', harm), ('thất bại, không cứu được', miss)]:
        for r in df.itertuples():
            text = SPLIT['test']['text'].get(r.text_id, r.text_id)
            DEMO_CASES[f'[{kind}] {text}  (CLIP {int(r.r_clip)} → {int(r.r_gat)})'] = r.text_id
print(len(DEMO_CASES), 'ca trong báo cáo | K* =', K_STAR)

# ---------- bộ mã hóa câu tự do: đúng CLIP ViT-B/32 của nhóm (kiểm tra khớp embedding đã trích) ----------
FREE_OK, FREE_MSG = False, ''
try:
    import open_clip
    clip_model, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai', device=DEVICE)
    tokenizer = open_clip.get_tokenizer('ViT-B-32'); clip_model.eval()
    @torch.no_grad()
    def encode_text(texts):
        return F.normalize(clip_model.encode_text(tokenizer(texts).to(DEVICE)).float(), dim=1).cpu()
    _chk = test_res.head(20)
    _mine = encode_text([SPLIT['test']['text'][str(t)] for t in _chk.text_id])
    _theirs = F.normalize(torch.stack([SPLIT['test']['tfeat'][SPLIT['test']['trow'][str(t)]].float() for t in _chk.text_id]), dim=1)
    FREE_OK = float((_mine * _theirs).sum(1).min()) >= 0.99
    FREE_MSG = '' if FREE_OK else 'Bộ mã hóa câu không khớp embedding của nhóm, nên tắt phần tự nhập.'
    IMG_EMB = F.normalize(torch.load(CLIP_ROOT / 'test/image_embeddings.pt', map_location='cpu', weights_only=True).float(), dim=1)
    _ii = pd.read_parquet(CLIP_ROOT / 'test/image_embedding_index.parquet')
    IMG_IDS = (_ii.sort_values('embedding_row') if 'embedding_row' in _ii else _ii).image_id.astype(str).tolist()
except Exception as exc:
    FREE_MSG = f'Không dùng được phần tự nhập: {exc}'
print('Tự nhập câu:', 'có' if FREE_OK else FREE_MSG)

# ---------- giải thích một ảnh trong top-50 ----------
def _rel_vi(r):
    return 'chứa' if r == 'contains' else r

def vg_graph(g, S, deltas):
    """Đồ thị S: nút ảnh (cam, vẽ bằng chính bức ảnh) và các vật; vật cùng tên gộp thành một nút (×n), quan hệ trùng cộng Δ."""
    lab = lambda v: 'ảnh' if v == g.root else node_label(g, v)
    seen_nodes = {}
    for u in S:
        for v in _endpoints(g, u):
            if v != g.root:
                seen_nodes.setdefault(lab(v), set()).add(v)
    items = merged_explanation(g, S, deltas, node_label, REL_NAME)
    vmax = max([t for _, _, t in items] + [1e-12]); size = Counter()
    for (s, r, d), n, t in items:
        for x in (s, d):
            if x != ROOT:
                size[x] += t
    nodes = {ROOT: W.node(ROOT, '', 'cand', 7, 'ảnh trả về')}
    for name, vs in seen_nodes.items():
        nm = name + (f' ×{len(vs)}' if len(vs) > 1 else '')
        nodes[name] = W.node(name, nm, 'other', 1.5 + 6 * size[name] / vmax, f'{nm} · tổng Δ = {size[name] * 1e3:.3g} × 10⁻³')
    links = []
    for (s, r, d), n, t in items:
        rv = _rel_vi(r) + (f' ×{n}' if n > 1 else '')
        sent = f'ảnh chứa {d}' if s == ROOT and r == 'contains' else f'{s} {r} {d}'
        links.append(W.link(s, d, rv, 'fact', t, f'{rv} ({t * 1e3:.3g})', f'{sent} · Δ = {t * 1e3:.3g} × 10⁻³'))
    story = None
    if items:
        (s, r, d), n, t = items[0]
        story = 'Bằng chứng mạnh nhất: ' + (f'ảnh chứa {d}' if s == ROOT and r == 'contains' else f'{s} {r} {d}') + (f' (×{n})' if n > 1 else '') + '.'
    return dict(nodes=list(nodes.values()), links=links), story

def vg_item(rank, j, ids, clip, gv, fused, qz, positive):
    img_id = ids[j]
    g = ComponentGraph(compact, image_to_component[img_id], image_features, entity_features, relation_type_features, DEVICE)
    m = explain_and_measure(SPEC['model'], g, qz, clip, gv, SPEC['alpha'], j, K_STAR, n_random=N_RANDOM, seed=SELECTION_SEED)
    it = dict(rank=rank, title=f'Ảnh {img_id}', correct=(img_id == positive) if positive else None,
              img=W.image_data_url(fetch_image(img_id)), row=dict(clip_rank=j + 1, removed=''),
              more=[('Hạng nếu chỉ dùng CLIP', str(j + 1)), ('Cosine CLIP', f'{float(clip[j]):.3f}'),
                    ('Điểm đồ thị', f'{float(gv[j]):+.3f}'), ('Điểm cuối', f'{float(fused[j]):+.3f}'),
                    ('Cách tính', f"{SPEC['alpha']:.1f}·z(CLIP) + {1 - SPEC['alpha']:.1f}·z(đồ thị) trên 50 ảnh CLIP, K* = {K_STAR}")])
    if not m.get('explained'):
        it['graph'] = dict(nodes=[], links=[])
        it['empty'] = 'Không quan hệ nào của ảnh làm tăng điểm đồ thị, nên không có giải thích S.'
        return it
    S = m['S']; deltas = dict(zip(S, m['deltas_S'])); r0 = m['rank0']; k = len(S)
    it['graph'], it['story'] = vg_graph(g, S, deltas)
    r_rm, r_rand = r0 + m['drank_rm'], r0 + m['drank_rand']
    it['why'] = [(f'Bỏ {k} quan hệ giải thích S', f'hạng {r0} → {W.rank_text(r_rm)}'),
                 (f'Bỏ {k} quan hệ ngẫu nhiên', f'hạng {r0} → {W.rank_text(r_rand)}')]
    it['removal'] = W.removal_sentence(k, 'quan hệ', r0, r_rm, r_rand)
    it['row']['removed'] = W.rank_text(r_rm)
    it['more'] += [('Fid+ của ảnh này', f"{m['fid_plus']:.3f} (bỏ ngẫu nhiên: {m['fid_plus_rand']:.3f})"),
                   ('Số quan hệ của ảnh', f"{m['n_units']} (S giữ {k}, sparsity {m['sparsity']:.2f})")]
    return it

LEGEND = {'cand': 'ảnh trả về', 'other': 'vật trong ảnh (to theo Δ)', 'fact': 'quan hệ trong giải thích S (dày theo Δ, số = Δ × 10⁻³)'}
ROLES = {'cand': 'ảnh hạng'}

def _page(text, ids, clip, gv, fused, qz, positive, target, k):
    order = torch.argsort(fused, descending=True).tolist()[:int(k)]
    items = [vg_item(r, j, ids, clip, gv, fused, qz, positive) for r, j in enumerate(order, 1)]
    header = dict(query=text, chips=[], target=target)
    return W.results_iframe(header, items, legend=LEGEND, roles=ROLES, removed_col='Bỏ S → hạng')

_RESULT = {}
@torch.no_grad()
def run_case_tid(tid, k=5):
    key = ('case', tid, int(k))
    if key not in _RESULT:
        row = test_res.iloc[ROW_OF[tid]]
        ids, clip, gv, qz, fused, rank, crank, top = score_row('test', MAIN, row)
        pos = str(row.positive_image_id)
        target = (f'Ảnh đúng: CLIP hạng {crank} → CLIP + GAT hạng {rank}' if pos in ids
                  else f'Ảnh đúng: CLIP hạng {crank}, ngoài top-50 nên không được xếp hạng lại')
        _RESULT[key] = _page(SPLIT['test']['text'].get(tid, tid), ids, clip, gv, fused, qz, pos, target, k)
    return _RESULT[key]

def run_case(choice, k=5):
    tid = DEMO_CASES.get(choice)
    return run_case_tid(tid, k) if tid else W.message_html('Chọn một ca trong danh sách.')

_TEXT2ID = {}
@torch.no_grad()
def run_free(text, k=5):
    text = (text or '').strip()
    if not text:
        return W.message_html('Nhập câu truy vấn tiếng Anh.')
    if not _TEXT2ID:
        _TEXT2ID.update({v.strip(): t for t, v in SPLIT['test']['text'].items()})
    if text in _TEXT2ID and _TEXT2ID[text] in ROW_OF:           # câu trùng một truy vấn test: dùng đúng truy vấn đó để biết ảnh đúng
        return run_case_tid(_TEXT2ID[text], k)
    if not FREE_OK:
        return W.message_html(FREE_MSG)
    key = ('free', text, int(k))
    if key not in _RESULT:
        q = encode_text([text])[0]; clip_all = IMG_EMB @ q
        top50 = torch.topk(clip_all, 50).indices.tolist(); ids = [IMG_IDS[j] for j in top50]
        qz = SPEC['model'].encode_query(q.unsqueeze(0).to(DEVICE))[0]
        gv = (split_emb('test', MAIN)[[SPLIT['test']['row'][x] for x in ids]] @ qz.cpu()).float()
        clip = clip_all[top50]; fused = fuse(clip, gv, SPEC['alpha'])
        _RESULT[key] = _page(text, ids, clip, gv, fused, qz, None, None, k)
    return _RESULT[key]

# ---------- giao diện ----------
try:
    import gradio as gr
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio'], check=True); import gradio as gr
_cs = list(DEMO_CASES)
with gr.Blocks(title='Truy vấn đa phương thức có giải thích bằng đồ thị') as demo:
    gr.Markdown('### Cấp độ 1 · CLIP + GAT trên scene graph Visual Genome')
    with gr.Row(equal_height=True):
        case_in = gr.Dropdown(_cs, value=_cs[0] if _cs else None, label='Ca trong báo cáo (5 thành công, 5 thất bại)', scale=6)
        k_in = gr.Slider(3, 10, value=5, step=1, label='Top-k', scale=1)
    with gr.Accordion('Tự nhập truy vấn', open=False):
        with gr.Row(equal_height=True):
            text_in = gr.Textbox(label='Câu truy vấn (tiếng Anh)', placeholder='a man riding a horse on the beach', scale=6)
            free_btn = gr.Button('Tìm', variant='primary', scale=1)
    out = gr.HTML()
    case_in.change(run_case, [case_in, k_in], out); k_in.release(run_case, [case_in, k_in], out)
    free_btn.click(run_free, [text_in, k_in], out); text_in.submit(run_free, [text_in, k_in], out)
    demo.load(run_case, [case_in, k_in], out)

if os.environ.get('XAI_DEMO_LAUNCH', '1') == '1':
    W.launch(demo, 'google.colab' in sys.modules)